# Model 2 – Update Facts in Supabase (no re-embedding)

In [1]:
!pip -q install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import requests, pandas as pd
from google.colab import userdata
from supabase import create_client
URL, KEY = userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_SERVICE_KEY")
sb = create_client(URL, KEY)
facts = pd.read_csv("fact_bank.csv"); scores = pd.read_csv("livelihood_scores.csv")
facts["livelihood"] = facts["livelihood"].fillna("ALL")
facts["n_obs"] = facts["n"]
facts["small_group"] = (facts["n"] < 10) & (facts["evidence_type"] == "monitoring")
scores["viability_low"] = scores["score_p5"]; scores["viability_high"] = scores["score_p95"]; scores["confidence"] = scores["rank_stability"]
defs = requests.get(f"{URL}/rest/v1/", headers={"apikey": KEY, "Authorization": f"Bearer {KEY}"}).json()["definitions"]
cols = {t: set(defs[t]["properties"]) for t in ("livelihood_facts", "livelihood_scores")}
def clean(df, table):
    keep = [c for c in df.columns if c in cols[table]]
    return df[keep].astype(object).where(df[keep].notna(), None).to_dict("records")
print(len(facts), "facts;", len(scores), "scores")

871 facts; 225 scores


## Check ids match

In [3]:
cur = []
for i in range(0, 2000, 1000):
    cur += sb.table("livelihood_facts").select("fact_id,scope,livelihood,district").range(i, i + 999).execute().data
cur = pd.DataFrame(cur).sort_values("fact_id").reset_index(drop=True)
new = facts[["fact_id", "scope", "livelihood", "district"]].sort_values("fact_id").reset_index(drop=True)
same = len(cur) == len(new) and (cur.fillna("") == new.fillna("")).all().all()
print("ids and keys match:", same)
assert same, "Stop: table and new fact bank do not line up"

ids and keys match: True


## Update

In [4]:
rows = clean(facts, "livelihood_facts")
for r in rows:
    fid = r.pop("fact_id"); r.pop("scope", None); r.pop("livelihood", None); r.pop("district", None)
    sb.table("livelihood_facts").update(r).eq("fact_id", fid).execute()
sb.table("livelihood_scores").delete().neq("livelihood", "__none__").execute()
srows = clean(scores, "livelihood_scores")
for i in range(0, len(srows), 100):
    sb.table("livelihood_scores").insert(srows[i:i + 100]).execute()
print("updated")

updated


## Check

In [5]:
print(sb.table("livelihood_facts").select("swot", count="exact").neq("swot", "none").limit(1).execute().count, "labelled facts (expect 82)")
print(sb.table("livelihood_scores").select("livelihood", count="exact").limit(1).execute().count, "scores (expect 225)")
print(sb.table("livelihood_scores").select("livelihood,district,viability,viability_low,viability_high").eq("district", "Ampara").limit(3).execute().data)

82 labelled facts (expect 82)
225 scores (expect 225)
[{'livelihood': 'AGRICULTURE & PLANTATION CROPS', 'district': 'Ampara', 'viability': 54.6, 'viability_low': 51.6, 'viability_high': 57.6}, {'livelihood': 'LIVESTOCK & ANIMAL HUSBANDRY', 'district': 'Ampara', 'viability': 62.6, 'viability_low': 61.1, 'viability_high': 64.4}, {'livelihood': 'FOOD PROCESSING & PRODUCTION', 'district': 'Ampara', 'viability': 58.7, 'viability_low': 56.2, 'viability_high': 61.4}]
